##RQ1 - Obs 1 to 5

In [14]:
import os
import math
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, fisher_exact
from statsmodels.stats.proportion import proportion_confint, confint_proportions_2indep
from statsmodels.stats.contingency_tables import Table2x2

# =========================
# CONFIG
# =========================
STAGE3_PATH = r"C:\Android Mobile App\ICST2026_Ext\0_Data_Feb_10\run_metrics_v16_stage3_enhanced.csv"
STAGE4_PATH = r"C:\Android Mobile App\ICST2026_Ext\0_Data_Feb_10\run_workload_signature_v1.csv"

OUT_DIR = r"C:\Android Mobile App\ICST2026_Ext\0_Data_Feb_10\rq1_outputs_profile"
os.makedirs(OUT_DIR, exist_ok=True)

TRUSTWORTHY = {"success", "failure"}
NONTRUST = {"cancelled", "skipped"}

# Column candidates
CONCLUSION_COLS = ["run_conclusion", "conclusion", "final_conclusion"]
EVENT_COLS = ["event", "event_name", "trigger"]
ATTEMPT_COLS = ["run_attempt", "attempt", "run_attempt_number"]
STYLES_TEXT_COLS = ["styles", "execution_styles", "style_text"]  # <- your Stage3 has "styles"
FULLNAME_COLS = ["full_name", "repo_full_name", "repository"]
RUNID_COLS = ["run_id", "workflow_run_id", "id"]
EVID_COLS = ["workload_evidence_level", "evidence_level", "evidence_bucket"]

# Default controlled slice for Obs 1.3 (edit if needed)
SLICE_TRIGGER = "push"
SLICE_ATTEMPT = "1"                  # "1" or ">1"
SLICE_EVID_ALLOWED = {"B", "C"}      # {"B","C"} is typical

BASELINE_STYLE = "Community"

# =========================
# Helpers
# =========================
def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"Missing expected column. Tried {candidates}. Have {list(df.columns)[:80]}...")

def to_num(s):
    return pd.to_numeric(s, errors="coerce")

def wilson_ci(x, n, alpha=0.05):
    lo, hi = proportion_confint(count=x, nobs=n, alpha=alpha, method="wilson")
    return float(lo), float(hi)

def or_ci_2x2(a, b, c, d, alpha=0.05):
    tab = np.array([[a, b], [c, d]], dtype=float)
    t = Table2x2(tab)
    or_ = float(t.oddsratio)
    lo, hi = t.oddsratio_confint(alpha=alpha)
    return or_, float(lo), float(hi)

def rd_ci_2x2(a, b, c, d, alpha=0.05):
    n1 = a + b
    n0 = c + d
    rd = (a / n1) - (c / n0)
    lo, hi = confint_proportions_2indep(
        count1=a, nobs1=n1, count2=c, nobs2=n0,
        method="newcomb", compare="diff", alpha=alpha
    )
    return float(rd), float(lo), float(hi)

def restrict_outcome_universe(df, conclusion_col):
    concl = df[conclusion_col].astype(str).str.strip().str.lower()
    keep = concl.isin(TRUSTWORTHY.union(NONTRUST))
    out = df.loc[keep].copy()
    out["trustworthy"] = concl.isin(TRUSTWORTHY).astype(int)
    return out

def assoc_test_multigroup(df, group_col):
    ct = pd.crosstab(df[group_col], df["trustworthy"])
    for col in [0, 1]:
        if col not in ct.columns:
            ct[col] = 0
    ct = ct[[0, 1]]
    chi2, p, dof, exp = chi2_contingency(ct.values)
    return ct, float(chi2), float(p), int(dof), float(np.min(exp))

def pairwise_vs_baseline(df, group_col, baseline_value):
    rows = []
    base = df[df[group_col] == baseline_value]
    c = int(base["trustworthy"].sum())
    d = int((1 - base["trustworthy"]).sum())

    for g, sub in df.groupby(group_col):
        n = len(sub)
        a = int(sub["trustworthy"].sum())
        b = int((1 - sub["trustworthy"]).sum())

        p = a / n if n else math.nan
        lo, hi = wilson_ci(a, n) if n else (math.nan, math.nan)

        if g == baseline_value:
            rows.append({
                group_col: g, "n": n,
                "trust_pct": 100*p, "ci_lo": 100*lo, "ci_hi": 100*hi,
                "RD_pp": math.nan, "RD_lo": math.nan, "RD_hi": math.nan,
                "OR": math.nan, "OR_lo": math.nan, "OR_hi": math.nan
            })
            continue

        rd, rd_lo, rd_hi = rd_ci_2x2(a, b, c, d)
        or_, or_lo, or_hi = or_ci_2x2(a, b, c, d)

        rows.append({
            group_col: g, "n": n,
            "trust_pct": 100*p, "ci_lo": 100*lo, "ci_hi": 100*hi,
            "RD_pp": 100*rd, "RD_lo": 100*rd_lo, "RD_hi": 100*rd_hi,
            "OR": or_, "OR_lo": or_lo, "OR_hi": or_hi
        })

    out = pd.DataFrame(rows)
    cats = [baseline_value] + [x for x in out[group_col].unique() if x != baseline_value]
    out[group_col] = pd.Categorical(out[group_col], categories=cats, ordered=True)
    return out.sort_values(group_col).reset_index(drop=True)

def to_latex_style_rows(style_profile, style_col="style", baseline=BASELINE_STYLE):
    lines = []
    for _, r in style_profile.iterrows():
        style = r[style_col]
        n = int(r["n"])
        trust = f'{r["trust_pct"]:.2f}'
        ci = f'[{r["ci_lo"]:.2f}, {r["ci_hi"]:.2f}]'
        if style == baseline:
            rd = "--"
            or_ = "--"
        else:
            rd = f'{r["RD_pp"]:+.2f}'
            or_ = f'{r["OR"]:.2f} [{r["OR_lo"]:.2f}, {r["OR_hi"]:.2f}]'
        lines.append(f"{style} & {n} & {trust} {ci} & {rd} & {or_} \\\\")
    return "\n".join(lines)

# -------------------------
# Style parsing (match your pipeline conventions)
# -------------------------
def normalize_style(styles_text: str) -> str:
    """
    Derive a single canonical style label from Stage3 'styles' text.
    Adjust keywords here ONLY if your pipeline uses different tokens.
    """
    s = (styles_text or "").strip().lower()
    if not s:
        return "Other"
    # priority: explicit styles first
    if "third-party" in s or "third party" in s:
        return "Third-Party"
    if "real-devices" in s or "real devices" in s or "physical" in s:
        return "Real-Devices"
    if "custom" in s:
        return "Custom"
    if "community" in s:
        return "Community"
    return "Other"


def main():
    df0 = pd.read_csv(STAGE3_PATH)

    conclusion_col = pick_col(df0, CONCLUSION_COLS)
    event_col = pick_col(df0, EVENT_COLS)
    attempt_col = pick_col(df0, ATTEMPT_COLS)
    styles_text_col = pick_col(df0, STYLES_TEXT_COLS)
    full_col = pick_col(df0, FULLNAME_COLS)
    runid_col = pick_col(df0, RUNID_COLS)

    # Normalize context cols
    df0["event_norm"] = df0[event_col].astype(str).str.strip().str.lower()
    df0["attempt_group"] = np.where(to_num(df0[attempt_col]).fillna(0).astype(int) > 1, ">1", "1")

    # Instrumentation-run universe
    if "instru_job_count" not in df0.columns:
        raise KeyError("Expected instru_job_count in Stage3.")
    df0 = df0[to_num(df0["instru_job_count"]).fillna(0).astype(int) > 0].copy()

    # Join evidence from Stage4
    s4 = pd.read_csv(STAGE4_PATH)
    s4_full = pick_col(s4, FULLNAME_COLS)
    s4_run = pick_col(s4, RUNID_COLS)
    s4_evid = pick_col(s4, EVID_COLS)
    s4_small = s4[[s4_full, s4_run, s4_evid]].copy()
    s4_small.rename(columns={s4_full: full_col, s4_run: runid_col, s4_evid: "evidence_level"}, inplace=True)

    df0 = df0.merge(s4_small, on=[full_col, runid_col], how="left")
    df0["evidence_level"] = df0["evidence_level"].astype(str).str.strip().str.upper()

    # Restrict to outcome universe and label trustworthy
    df = restrict_outcome_universe(df0, conclusion_col)

    # Add derived style label
    df["style"] = df[styles_text_col].astype(str).apply(normalize_style)

    # =========================
    # Obs 1.1 overall
    # =========================
    n = len(df)
    x = int(df["trustworthy"].sum())
    p = x / n
    lo, hi = wilson_ci(x, n)
    overall = {"n": n, "trust_count": x, "trust_pct": 100*p, "ci_lo": 100*lo, "ci_hi": 100*hi}
    print("\n[Obs 1.1] Overall:", overall)
    pd.DataFrame([overall]).to_csv(os.path.join(OUT_DIR, "obs1_1_overall.csv"), index=False)

    # =========================
    # Obs 1.2 validated controls
    # =========================
    # trigger
    ct_event, chi2, pval, dof, minexp = assoc_test_multigroup(df, "event_norm")
    print("\n[Obs 1.2] Trigger chi2:", chi2, "p:", pval, "dof:", dof, "minexp:", minexp)
    ct_event.to_csv(os.path.join(OUT_DIR, "obs1_2_trigger_contingency.csv"))

    trig_tbl = pairwise_vs_baseline(df, "event_norm", baseline_value="push")
    trig_tbl.to_csv(os.path.join(OUT_DIR, "obs1_2_trigger_effects.csv"), index=False)

    # attempt (Fisher)
    ct_attempt = pd.crosstab(df["attempt_group"], df["trustworthy"])
    for col in [0, 1]:
        if col not in ct_attempt.columns:
            ct_attempt[col] = 0
    ct_attempt = ct_attempt[[0, 1]]
    a = int(ct_attempt.loc[">1", 1]) if ">1" in ct_attempt.index else 0
    b = int(ct_attempt.loc[">1", 0]) if ">1" in ct_attempt.index else 0
    c = int(ct_attempt.loc["1", 1]) if "1" in ct_attempt.index else 0
    d = int(ct_attempt.loc["1", 0]) if "1" in ct_attempt.index else 0
    _, p_fisher = fisher_exact([[a, b], [c, d]], alternative="two-sided")
    print("\n[Obs 1.2] Attempt Fisher p:", p_fisher)
    ct_attempt.to_csv(os.path.join(OUT_DIR, "obs1_2_attempt_contingency.csv"))

    att_tbl = pairwise_vs_baseline(df, "attempt_group", baseline_value="1")
    att_tbl.to_csv(os.path.join(OUT_DIR, "obs1_2_attempt_effects.csv"), index=False)

    # evidence
    evid = df[df["evidence_level"].isin(["A", "B", "C"])].copy()
    ct_evid, chi2e, pvale, dofe, minexpe = assoc_test_multigroup(evid, "evidence_level")
    print("\n[Obs 1.2] Evidence chi2:", chi2e, "p:", pvale, "dof:", dofe, "minexp:", minexpe)
    ct_evid.to_csv(os.path.join(OUT_DIR, "obs1_2_evidence_contingency.csv"))

    evid_tbl = pairwise_vs_baseline(evid, "evidence_level", baseline_value="C")
    evid_tbl.to_csv(os.path.join(OUT_DIR, "obs1_2_evidence_effects.csv"), index=False)

    # =========================
    # Obs 1.3 controlled style reliability profile
    # =========================
    prof = df[
        (df["event_norm"] == SLICE_TRIGGER) &
        (df["attempt_group"] == SLICE_ATTEMPT) &
        (df["evidence_level"].isin(SLICE_EVID_ALLOWED))
    ].copy()

    if len(prof) == 0:
        raise RuntimeError("Controlled slice produced 0 rows. Check SLICE_* settings.")

    if BASELINE_STYLE not in set(prof["style"]):
        raise RuntimeError(f"Baseline style '{BASELINE_STYLE}' not present in controlled slice.")

    style_profile = pairwise_vs_baseline(prof, "style", baseline_value=BASELINE_STYLE)
    style_profile.to_csv(os.path.join(OUT_DIR, "obs1_3_style_profile_controlled.csv"), index=False)

    latex_rows = to_latex_style_rows(style_profile, style_col="style", baseline=BASELINE_STYLE)
    with open(os.path.join(OUT_DIR, "obs1_3_style_profile_controlled_rows.tex"), "w", encoding="utf-8") as f:
        f.write(latex_rows + "\n")

    print("\n[Obs 1.3] Controlled style reliability profile (slice:",
          f"trigger={SLICE_TRIGGER}, attempt={SLICE_ATTEMPT}, evidence={sorted(SLICE_EVID_ALLOWED)})")
    print(style_profile[["style","n","trust_pct","ci_lo","ci_hi","RD_pp","OR","OR_lo","OR_hi"]])

    print("\n[done] Outputs written to:", OUT_DIR)


if __name__ == "__main__":
    main()



[Obs 1.1] Overall: {'n': 6831, 'trust_count': 6105, 'trust_pct': 89.3719806763285, 'ci_lo': 88.61886475030991, 'ci_hi': 90.08083929415098}

[Obs 1.2] Trigger chi2: 23.359030568529292 p: 3.398924199610625e-05 dof: 3 minexp: 2.3381642512077296

[Obs 1.2] Attempt Fisher p: 0.00010116747358418348

[Obs 1.2] Evidence chi2: 243.9432659299863 p: 1.067561380421249e-53 dof: 2 minexp: 37.091787439613526

[Obs 1.3] Controlled style reliability profile (slice: trigger=push, attempt=1, evidence=['B', 'C'])
         style     n   trust_pct      ci_lo       ci_hi      RD_pp         OR  \
0    Community  5829   88.265569  87.414069   89.066665        NaN        NaN   
1        Other    13  100.000000  77.190463  100.000000  11.734431   3.456560   
2  Third-Party   136  100.000000  97.252990  100.000000  11.734431  36.160933   

      OR_lo       OR_hi  
0       NaN         NaN  
1  0.204853   58.323706  
2  2.247763  581.739630  

[done] Outputs written to: C:\Android Mobile App\ICST2026_Ext\0_Data_F

In [12]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats

# =========================
# SET YOUR DATA FOLDER HERE
# =========================
ROOT_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext\0_Data_Feb_10")

# Expected filenames (tries in order)
STAGE3_CANDIDATES = [
    "run_metrics_v16_stage3_enhanced.csv",
    "run_metrics_v16_from_verified_workflows_with_env_style.csv",
]
STEPS_CANDIDATES = [
    "run_steps_v16_stage3_breakdown.csv",
    "run_steps_v16_stage3_breakdown_with_overhead_group.csv",
]
STAGE4_CANDIDATES = [
    "run_workload_signature_v1.csv",
]

def pick_existing(root: Path, names):
    for n in names:
        p = root / n
        if p.exists():
            return p
    raise FileNotFoundError(f"None of these files exist in {root}:\n  - " + "\n  - ".join(names))

STAGE3_PATH = pick_existing(ROOT_DIR, STAGE3_CANDIDATES)
STEPS_PATH  = pick_existing(ROOT_DIR, STEPS_CANDIDATES)
STAGE4_PATH = pick_existing(ROOT_DIR, STAGE4_CANDIDATES)

print("[debug] Using Stage3:", STAGE3_PATH)
print("[debug] Using Steps :", STEPS_PATH)
print("[debug] Using Stage4:", STAGE4_PATH)

# =========================
# Helpers
# =========================
def style_bucket(s: str) -> str:
    s = str(s or "").lower()
    if "real" in s:
        return "Real-Devices"
    if "third" in s:
        return "Third-Party"
    if "gmd" in s:
        return "GMD"
    if "emu_custom" in s or "custom" in s:
        return "Custom"
    if "emu_community" in s or "community" in s:
        return "Community"
    return "Other"

def med_iqr_p95(x):
    x = pd.to_numeric(x, errors="coerce").dropna().to_numpy()
    if len(x) == 0:
        return (0, np.nan, np.nan, np.nan, np.nan)
    med = np.median(x)
    q1  = np.percentile(x, 25)
    q3  = np.percentile(x, 75)
    p95 = np.percentile(x, 95)
    return (len(x), med, q1, q3, p95)

def fmt_median_iqr(m, q1, q3):
    return f"{m:.0f} [{q1:.0f}, {q3:.0f}]"

def cliffs_delta(a, b):
    a = pd.to_numeric(pd.Series(a), errors="coerce").dropna().to_numpy()
    b = pd.to_numeric(pd.Series(b), errors="coerce").dropna().to_numpy()
    if len(a) == 0 or len(b) == 0:
        return np.nan
    U = stats.mannwhitneyu(a, b, alternative="two-sided").statistic
    return float(2 * U / (len(a) * len(b)) - 1)

def bootstrap_ci_delta(a, b, boots=300, seed=1):
    rng = np.random.default_rng(seed)
    a = pd.to_numeric(pd.Series(a), errors="coerce").dropna().to_numpy()
    b = pd.to_numeric(pd.Series(b), errors="coerce").dropna().to_numpy()
    if len(a) == 0 or len(b) == 0:
        return (np.nan, np.nan, np.nan)
    deltas = []
    for _ in range(boots):
        aa = rng.choice(a, size=len(a), replace=True)
        bb = rng.choice(b, size=len(b), replace=True)
        deltas.append(cliffs_delta(aa, bb))
    lo, hi = np.percentile(deltas, [2.5, 97.5])
    return (cliffs_delta(a, b), float(lo), float(hi))

def holm_adjust(pvals):
    pvals = np.array(pvals, dtype=float)
    m = len(pvals)
    order = np.argsort(pvals)
    adj = np.empty(m)
    for i, idx in enumerate(order):
        adj[idx] = min(1.0, (m - i) * pvals[idx])
    for i in range(m - 2, -1, -1):
        adj[order[i]] = min(adj[order[i]], adj[order[i + 1]])
    return adj

# =========================
# Load data
# =========================
df3 = pd.read_csv(STAGE3_PATH)
df4 = pd.read_csv(STAGE4_PATH)
steps = pd.read_csv(STEPS_PATH)

print("[debug] Stage3 rows:", len(df3), "cols:", len(df3.columns))
print("[debug] Stage4 rows:", len(df4), "cols:", len(df4.columns))
print("[debug] Steps rows :", len(steps), "cols:", len(steps.columns))

# Columns used
NEEDED_STAGE4 = ["full_name", "run_id", "workload_evidence_level"]
missing4 = [c for c in NEEDED_STAGE4 if c not in df4.columns]
if missing4:
    raise KeyError(f"Stage4 is missing required columns: {missing4}. Available: {list(df4.columns)[:50]}")

df4 = df4[NEEDED_STAGE4].copy()

# numeric conversions
for c in ["instru_job_count", "run_duration_seconds", "queue_seconds",
          "ttfts_seconds", "time_to_first_instru_seconds"]:
    if c in df3.columns:
        df3[c] = pd.to_numeric(df3[c], errors="coerce")

# =========================
# Instrumentation subset (aligned with RQ1)
# =========================
if "instru_job_count" not in df3.columns:
    raise KeyError("Stage3 is missing instru_job_count. Check your Stage3 file choice.")

df = df3[df3["instru_job_count"].fillna(0) > 0].copy()
df["style"] = df["styles"].apply(style_bucket)

# join stage4 evidence
df = df.merge(df4, on=["full_name", "run_id"], how="left")

# Trustworthy subset for speed
if "run_conclusion" not in df.columns:
    raise KeyError("Stage3 is missing run_conclusion.")
df = df[df["run_conclusion"].isin(["success", "failure"])].copy()

# TTFS: prefer step-based ttfts_seconds else fallback proxy
if "ttfts_seconds" not in df.columns:
    df["ttfts_seconds"] = np.nan
if "time_to_first_instru_seconds" not in df.columns:
    df["time_to_first_instru_seconds"] = np.nan
df["ttfs_final"] = df["ttfts_seconds"].combine_first(df["time_to_first_instru_seconds"])

print("[debug] trustworthy instrumentation runs n =", len(df))
print("[debug] style counts:\n", df["style"].value_counts())
print("[debug] trigger counts:\n", df["event"].value_counts())
print("[debug] evidence counts:\n", df["workload_evidence_level"].value_counts(dropna=False))

# =========================
# Obs 2.1: Duration by style + key pairwise vs Community
# =========================
styles_order = ["Community", "Custom", "GMD", "Third-Party", "Real-Devices"]
rows = []
for st in styles_order:
    sub = df[df["style"] == st]
    n, med, q1, q3, p95 = med_iqr_p95(sub["run_duration_seconds"])
    if n == 0:
        continue
    rows.append([st, int(n), fmt_median_iqr(med, q1, q3), float(p95)])
dur_by_style = pd.DataFrame(rows, columns=["style", "n", "median_iqr", "p95"])
print("\n[Obs2.1] duration_by_style:\n", dur_by_style)

base = df[df["style"] == "Community"]["run_duration_seconds"].dropna()
pair_stats = []
pvals = []
for st in ["Third-Party", "Real-Devices", "Custom", "GMD"]:
    b = df[df["style"] == st]["run_duration_seconds"].dropna()
    if len(b) == 0:
        continue
    p = stats.mannwhitneyu(base, b, alternative="two-sided").pvalue
    d, lo, hi = bootstrap_ci_delta(base, b, boots=300, seed=2)
    pair_stats.append([st, len(b), p, d, lo, hi])
    pvals.append(p)
if pair_stats:
    adj = holm_adjust(pvals)
    for i in range(len(pair_stats)):
        pair_stats[i].insert(3, adj[i])
dur_pairwise = pd.DataFrame(
    pair_stats, columns=["style_vs_comm", "n", "p_raw", "p_holm", "cliffs_d", "d_ci_lo", "d_ci_hi"]
)
print("\n[Obs2.1] pairwise vs Community:\n", dur_pairwise)

# =========================
# Obs 2.2: Queue presence by trigger + chi2 + queue stats among queue-positive
# =========================
if "queue_seconds" not in df.columns:
    df["queue_seconds"] = np.nan
df["queue_pos"] = df["queue_seconds"].fillna(0) > 0

ct = pd.crosstab(df["event"], df["queue_pos"])
if ct.shape[1] == 1:
    # all True or all False edge case
    ct[~ct.columns[0]] = 0
chi2, p, dof, _ = stats.chi2_contingency(ct.values)
print("\n[Obs2.2] queue_pos contingency:\n", ct)
print("[Obs2.2] chi2=", chi2, "p=", p, "dof=", dof)

qrows = []
for ev, sub in df.groupby("event"):
    n = len(sub)
    rate = sub["queue_pos"].mean() * 100
    subpos = sub[sub["queue_pos"]]["queue_seconds"].dropna()
    if len(subpos) > 0:
        med = float(np.median(subpos))
        p95q = float(np.percentile(subpos, 95))
    else:
        med = np.nan
        p95q = np.nan
    qrows.append([ev, n, rate, med, p95q])
queue_by_trigger = pd.DataFrame(
    qrows, columns=["event", "n", "queue_pos_pct", "queue_med_pos", "queue_p95_pos"]
).sort_values("event")
print("\n[Obs2.2] queue_by_trigger:\n", queue_by_trigger)

# =========================
# Obs 2.3: TTFS by style + key pairwise Community vs Third-Party
# =========================
rows = []
for st in styles_order:
    sub = df[df["style"] == st]
    n, med, q1, q3, p95 = med_iqr_p95(sub["ttfs_final"])
    if n == 0:
        continue
    rows.append([st, int(n), fmt_median_iqr(med, q1, q3), float(p95)])
ttfs_by_style = pd.DataFrame(rows, columns=["style", "n", "median_iqr", "p95"])
print("\n[Obs2.3] ttfs_by_style:\n", ttfs_by_style)

a = df[df["style"] == "Community"]["ttfs_final"].dropna()
b = df[df["style"] == "Third-Party"]["ttfs_final"].dropna()
if len(a) > 0 and len(b) > 0:
    p_tt = stats.mannwhitneyu(a, b, alternative="two-sided").pvalue
    d_tt, lo_tt, hi_tt = bootstrap_ci_delta(a, b, boots=300, seed=3)
    print("\n[Obs2.3] Community vs Third-Party: p=", p_tt, "Cliff_d=", d_tt, "CI=[", lo_tt, ",", hi_tt, "]")

# =========================
# Obs 2.4: Phase shares from steps (Stage 3B)
# =========================
# Expect step file to have duration_seconds + category + full_name + run_id
for col in ["full_name", "run_id", "category"]:
    if col not in steps.columns:
        raise KeyError(f"Steps file missing '{col}'. Available: {list(steps.columns)[:50]}")
if "duration_seconds" not in steps.columns:
    # sometimes stage3b uses different naming; try to find it
    cand = [c for c in steps.columns if "duration" in c.lower() and "second" in c.lower()]
    if not cand:
        raise KeyError("Steps file missing duration_seconds (or similar).")
    steps = steps.rename(columns={cand[0]: "duration_seconds"})

steps["duration_seconds"] = pd.to_numeric(steps["duration_seconds"], errors="coerce").fillna(0)

key = df[["full_name", "run_id", "style"]].copy()
s = steps.merge(key, on=["full_name", "run_id"], how="inner")

pv = s.pivot_table(
    index=["full_name", "run_id", "style"],
    columns="category",
    values="duration_seconds",
    aggfunc="sum",
    fill_value=0,
).reset_index()

# normalize expected categories
for col in ["artifact", "third_party", "env_setup", "test", "other"]:
    if col not in pv.columns:
        pv[col] = 0.0

pv["prov_art"] = pv["artifact"] + pv["third_party"]
pv["setup"] = pv["env_setup"]
pv["test_time"] = pv["test"]
pv["other_time"] = pv["other"]
pv["measured_total"] = pv[["setup", "test_time", "prov_art", "other_time"]].sum(axis=1)

for c in ["setup", "test_time", "prov_art", "other_time"]:
    pv[c + "_share"] = np.where(pv["measured_total"] > 0, pv[c] / pv["measured_total"] * 100, np.nan)

phase_rows = []
for st in ["Community", "Third-Party", "Real-Devices", "Custom", "GMD"]:
    sub = pv[pv["style"] == st]
    if len(sub) == 0:
        continue
    phase_rows.append([
        st, len(sub),
        float(np.nanmedian(sub["setup_share"])),
        float(np.nanmedian(sub["test_time_share"])),
        float(np.nanmedian(sub["prov_art_share"])),
        float(np.nanmedian(sub["other_time_share"])),
    ])
phase_by_style = pd.DataFrame(
    phase_rows,
    columns=["style", "n", "setup_pct", "test_pct", "prov_art_pct", "other_pct"]
)
print("\n[Obs2.4] phase_by_style:\n", phase_by_style)

# Kruskal (Community vs Third-Party) as key contrast
setup_comm = pv[pv["style"]=="Community"]["setup_share"].dropna()
setup_tp   = pv[pv["style"]=="Third-Party"]["setup_share"].dropna()
prov_comm  = pv[pv["style"]=="Community"]["prov_art_share"].dropna()
prov_tp    = pv[pv["style"]=="Third-Party"]["prov_art_share"].dropna()

if len(setup_comm)>0 and len(setup_tp)>0:
    H_setup, p_setup = stats.kruskal(setup_comm, setup_tp)
else:
    p_setup = np.nan
if len(prov_comm)>0 and len(prov_tp)>0:
    H_prov, p_prov = stats.kruskal(prov_comm, prov_tp)
else:
    p_prov = np.nan
print("\n[Obs2.4] setup_share kruskal p=", p_setup, "| prov_art_share kruskal p=", p_prov)

# =========================
# Obs 2.5: Duration by evidence level + Kruskal
# =========================
erows = []
for lvl in ["A", "B", "C"]:
    sub = df[df["workload_evidence_level"] == lvl]
    n, med, q1, q3, p95 = med_iqr_p95(sub["run_duration_seconds"])
    erows.append([lvl, int(n), fmt_median_iqr(med, q1, q3), float(p95)])
dur_by_evidence = pd.DataFrame(erows, columns=["evidence", "n", "median_iqr", "p95"])
print("\n[Obs2.5] duration_by_evidence:\n", dur_by_evidence)

gA = df[df["workload_evidence_level"]=="A"]["run_duration_seconds"].dropna()
gB = df[df["workload_evidence_level"]=="B"]["run_duration_seconds"].dropna()
gC = df[df["workload_evidence_level"]=="C"]["run_duration_seconds"].dropna()
if len(gA)>0 and len(gB)>0 and len(gC)>0:
    H_e, p_e = stats.kruskal(gA, gB, gC)
else:
    p_e = np.nan
print("[Obs2.5] Kruskal p=", p_e)

# =========================
# Emit LaTeX tables
# =========================
print("\n--- LaTeX: Obs2.1 duration_by_style ---")
print(dur_by_style.to_latex(index=False, escape=False))

print("\n--- LaTeX: Obs2.2 queue_by_trigger ---")
print(queue_by_trigger.to_latex(index=False, escape=False))

print("\n--- LaTeX: Obs2.3 ttfs_by_style ---")
print(ttfs_by_style.to_latex(index=False, escape=False))

print("\n--- LaTeX: Obs2.4 phase_by_style ---")
print(phase_by_style.to_latex(index=False, escape=False))

print("\n--- LaTeX: Obs2.5 duration_by_evidence ---")
print(dur_by_evidence.to_latex(index=False, escape=False))


[debug] Using Stage3: C:\Android Mobile App\ICST2026_Ext\0_Data_Feb_10\run_metrics_v16_stage3_enhanced.csv
[debug] Using Steps : C:\Android Mobile App\ICST2026_Ext\0_Data_Feb_10\run_steps_v16_stage3_breakdown.csv
[debug] Using Stage4: C:\Android Mobile App\ICST2026_Ext\0_Data_Feb_10\run_workload_signature_v1.csv
[debug] Stage3 rows: 31591 cols: 49
[debug] Stage4 rows: 31591 cols: 23
[debug] Steps rows : 517797 cols: 14
[debug] trustworthy instrumentation runs n = 6105
[debug] style counts:
 style
Community       5949
Third-Party      138
Real-Devices      12
Custom             5
Other              1
Name: count, dtype: int64
[debug] trigger counts:
 event
push                 5760
schedule              177
pull_request          148
workflow_dispatch      20
Name: count, dtype: int64
[debug] evidence counts:
 workload_evidence_level
C    3162
B    2599
A     344
Name: count, dtype: int64

[Obs2.1] duration_by_style:
           style     n         median_iqr     p95
0     Community  5949